# TCC WBE-IoT — Avaliação final de 2024

Este notebook congela formalmente o Poisson e abre o holdout de 2024 **uma única vez**. Ele não seleciona modelos. Execute as células em ordem e leia o protocolo antes da confirmação.

## 1. Instalar dependências

In [ ]:
%pip -q install "pandas==2.2.3" "numpy==2.1.3" "pyarrow==20.0.0" "scikit-learn==1.6.1" "matplotlib>=3.8,<4" "joblib>=1.4,<2"

## 2. Clonar ou atualizar a branch de modelagem

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git"
GIT_REF = "feature/pipeline-modelagem"
REPO_DIR = Path("/content/tcc-vigilancia-wbe-iot")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{GIT_REF}"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())

## 3. Montar o Drive e validar o snapshot oficial

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = Path("/content/drive/MyDrive/TCC/datasets/dataset_modelagem_v1_20260927_20261006T182348Z.parquet")
RUNS_DIR = Path("/content/drive/MyDrive/TCC/final_evaluation")
EXPECTED_SHA256 = "1d51c54f1d13e28b9253b6db4fa12d9538ed774da6c5d5c33b52f2cd026fcb85"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_PATH.exists(), f"Dataset não encontrado: {DATASET_PATH}"

import hashlib
digest = hashlib.sha256()
with DATASET_PATH.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f"SHA-256 divergente: {actual_sha256}"
print("Dataset oficial validado:", actual_sha256)

## 4. Fase 1 — congelar o protocolo

Esta fase registra modelo, features, hiperparâmetros e regra de alerta. Ela não calcula métricas de 2024.

In [ ]:
import os
import sys

before = {path for path in RUNS_DIR.iterdir() if path.is_dir()}
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR)
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts/run_final_evaluation.py"), "freeze", str(DATASET_PATH), "--output", str(RUNS_DIR)],
    cwd=REPO_DIR, env=env, check=True,
)
after = {path for path in RUNS_DIR.iterdir() if path.is_dir()}
new_runs = sorted(after - before)
assert len(new_runs) == 1, f"Não foi possível identificar o congelamento: {new_runs}"
RUN_DIR = new_runs[0]
print("Protocolo salvo em:", RUN_DIR)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RUN_DIR / "00_LEIA_ANTES.md").read_text(encoding="utf-8")))

## 5. Pausa metodológica obrigatória

Confira o protocolo exibido acima. A próxima fase abrirá 2024 e impedirá uma segunda execução para este dataset. O resultado não poderá ser usado para trocar o modelo.

In [ ]:
# Depois de conferir o protocolo, substitua o texto vazio pelo token exibido abaixo.
CONFIRMAR_ABERTURA_2024 = ""
TOKEN_ESPERADO = "ABRIR_2024_APENAS_PARA_TESTE_FINAL"
print("Token esperado:", TOKEN_ESPERADO)

## 6. Fase 2 — abrir 2024 e executar o teste final

In [ ]:
assert CONFIRMAR_ABERTURA_2024 == TOKEN_ESPERADO, "Confirmação ausente: 2024 não foi aberto."
subprocess.run(
    [
        sys.executable, str(REPO_DIR / "scripts/run_final_evaluation.py"),
        "evaluate", str(DATASET_PATH),
        "--run-dir", str(RUN_DIR),
        "--confirm-open-2024", CONFIRMAR_ABERTURA_2024,
    ],
    cwd=REPO_DIR, env=env, check=True,
)

## 7. Conferir os resultados finais

In [ ]:
import pandas as pd
from IPython.display import Image

display(Markdown((RUN_DIR / "RESUMO_EXECUCAO.md").read_text(encoding="utf-8")))
display(pd.read_csv(RUN_DIR / "01_metricas_teste_2024.csv"))
display(pd.read_csv(RUN_DIR / "02_metricas_alerta_2024.csv"))
display(Image(filename=str(RUN_DIR / "04_observado_vs_previsto_2024.png")))

## 8. Compactar os artefatos

In [ ]:
import shutil
zip_path = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("ZIP criado:", zip_path)
print("Envie este ZIP para auditoria. Não reexecute a fase 2.")